# Day 02 — A Tour of Spark's Toolset
**Book:** Spark: The Definitive Guide | **Chapter:** 3  
**Time estimate:** 60–75 min

---

## What you'll learn today
- Running Spark as a standalone application (`spark-submit`) vs interactively
- A first look at each major Spark module: Datasets, Structured Streaming, MLlib, Lower-Level APIs (RDDs)
- **Spark UI deep dive #1**: the Jobs and Stages tabs, and what a "job" actually corresponds to in your code
- How to read the **DAG visualization** for a multi-stage query
- Caching and why it changes the DAG

> This is a "tour" chapter in the book — breadth over depth. We'll go broad here too, but anchor every concept to the Spark UI so you build the habit of *looking* before *guessing*.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

def get_spark(app_name: str = "spark_study", shuffle_partitions: int = 4) -> SparkSession:
    return (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.shuffle.partitions", str(shuffle_partitions))
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )

spark = get_spark("day02_toolset")
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ Spark {spark.version} ready")
print(f"   Spark UI: http://localhost:4040")
print()
print("👉 Open the UI now and keep it open in a side window for this whole notebook.")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/25 11:18:47 WARN Utils: Your hostname, jitendrarawatbitsinglasscoms-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.27 instead (on interface en0)
26/06/25 11:18:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 11:18:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 4.1.2 ready
   Spark UI: http://localhost:4040

👉 Open the UI now and keep it open in a side window for this whole notebook.


---
## 1. Interactive vs Production: `spark-submit`

So far we've run Spark interactively (notebook = driver process stays alive, REPL-style).  
In production, you package your code and submit it as a **job** via `spark-submit`:

```bash
spark-submit \
  --master local[*] \
  --class com.example.MyApp \
  my_script.py
```

For Python, it's simply:
```bash
spark-submit --master local[*] my_script.py
```

### Key difference from notebooks
| | Notebook (interactive) | `spark-submit` (production) |
|-|------------------------|------------------------------|
| SparkSession lifetime | Stays alive across cells | Created once, ends when script ends |
| UI availability | localhost:4040 while running | Same, but UI dies when job finishes (use History Server to view later) |
| Use case | Exploration, development | Scheduled jobs, pipelines |

We'll build a real `spark-submit` script in **Day 15 (Developing Spark Applications)**. For now, just know: everything you write in a notebook can become a `.py` script with almost no changes — the SparkSession boilerplate is identical.


---
## 2. Datasets — A Quick Preview

The book introduces three core structured concepts:

| Concept | What it is |
|---------|-----------|
| **DataFrame** | Distributed table of `Row` objects — Python's primary structured API |
| **Dataset** | Typed version of DataFrame — **JVM-only** (Scala/Java). Not available in PySpark. |
| **SQL Table/View** | A named reference to a DataFrame, queryable via SQL |

### Why mention Datasets if PySpark doesn't have them?
Because **`DataFrame` in Python is actually `Dataset[Row]`** under the hood — `Row` is a generic, untyped container. In Scala, you can have `Dataset[Person]` with compile-time type checking. In Python, you get the same execution engine but with dynamic typing.

**Practical implication:** when you see Scala Spark code using `.as[Person]` or `Dataset[T]`, know that the Python equivalent is just a `DataFrame` with a schema — the runtime behaviour (Catalyst, Tungsten) is identical. We'll fully cover this in **Day 10 (Datasets API)**.


In [3]:
# Demonstrate: Python DataFrame == Dataset[Row] conceptually
data = [("Alice", 30), ("Bob", 25), ("Carol", 35)]
df = spark.createDataFrame(data, schema=["name", "age"])

# Every row is a Row object - a generic, dynamically-typed container
row = df.first()
print(f"Type of a row: {type(row)}")
print(f"Row contents: {row}")
print(f"Access by name: {row.name}, {row.age}")
print(f"Access by index: {row[0]}, {row[1]}")

# This dynamic Row is what makes PySpark "untyped" compared to Scala Datasets


Type of a row: <class 'pyspark.sql.types.Row'>
Row contents: Row(name='Alice', age=30)
Access by name: Alice, 30
Access by index: Alice, 30


---
## 3. 🔍 Spark UI Deep Dive #1 — Jobs, Stages, and the DAG

This is the first of our recurring Spark UI deep dives. Today: **Jobs** and **Stages**.

### The hierarchy
```
Application (one SparkSession)
  └── Job (one per action: count, show, collect, write...)
        └── Stage (one per shuffle boundary)
              └── Task (one per partition)
```

### What triggers a "Job"?
**Every action** creates exactly one job. If you call `.count()` then `.show()` on the same DataFrame, that's **2 jobs** — even though the transformations are identical, Spark re-executes (unless cached).

### What you'll do in this section
1. Run a query with a deliberate **2-stage shape** (one shuffle)
2. Open the UI's **Jobs** tab — find the job
3. Click into it — see **2 stages**
4. Click into each stage — see **tasks**, and crucially, **shuffle read/write bytes**
5. Go to the **SQL** tab — see the same query as a visual DAG diagram

Run the cell below, then follow the UI walkthrough in the markdown cell after it.


In [7]:
# A query with exactly ONE shuffle (one wide transformation) → 2 stages

sales = spark.createDataFrame([
    ("Toronto",  "Electronics", 1200),
    ("Toronto",  "Clothing",     300),
    ("Montreal", "Electronics",  900),
    ("Montreal", "Clothing",     450),
    ("Calgary",  "Electronics",  700),
    ("Calgary",  "Clothing",     250),
    ("Toronto",  "Electronics",  800),
    ("Montreal", "Clothing",     150),
], schema=["city", "category", "revenue"])

# Stage 1 (narrow): filter
# Stage 1→2 boundary (wide): groupBy triggers a shuffle
# Stage 2 (after shuffle): aggregate + sort
city_totals = (
    sales
    .filter(F.col("revenue") > 100)         # narrow — stays in stage 1
    .groupBy("city")                          # ⚡ SHUFFLE — stage boundary
    .agg(F.sum("revenue").alias("total_revenue"))
    .orderBy(F.col("total_revenue").desc())   # same stage as agg (no extra shuffle needed here)
)

city_totals.show()
print("✅ Job complete. Now go to the Spark UI.")


+--------+-------------+
|    city|total_revenue|
+--------+-------------+
| Toronto|         2300|
|Montreal|         1500|
| Calgary|          950|
+--------+-------------+

✅ Job complete. Now go to the Spark UI.


In [8]:
city_totals.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [3]: [city#46, category#47, revenue#48L]
Arguments: [city#46, category#47, revenue#48L], MapPartitionsRDD[52] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [3]: [city#46, category#47, revenue#48L]
Condition : (isnotnull(revenue#48L) AND (revenue#48L > 100))

(3) Project
Output [2]: [city#46, revenue#48L]
Input [3]: [city#46, category#47, revenue#48L]

(4) HashAggregate
Input [2]: [city#46, revenue#48L]
Keys [1]: [city#46]
Functions [1]: [partial_sum(revenue#48L)]
Aggregate Attributes [1]: [sum#56L]
Results [2]: [city#46, sum#57L]

(5) Exchange
Input [2]: [city#46, sum#57L]
Arguments: hashpartitioning(city#46, 4), ENSURE_REQUIRE

### 👉 UI Walkthrough — do this now

1. **Jobs tab** (`http://localhost:4040/jobs/`)
   - You should see a job with description containing `showString` (triggered by `.show()`)
   - Note the **"Stages: 2/2"** — confirms our prediction of 2 stages

2. **Click the job** → see the **stage list**
   - **Stage 1**: `Scan ... → Filter → ...` — this ran the `filter()`. Check "Shuffle Write" column — should show non-zero bytes (data being written for the next stage to read)
   - **Stage 2**: `... → HashAggregate → Sort` — check "Shuffle Read" — should roughly match Stage 1's write

3. **Click into Stage 1** → **Tasks table**
   - How many tasks ran? (Should match `sales.rdd.getNumPartitions()` — run the cell below to check)
   - Look at "Duration" column — are tasks roughly equal? (They should be, this is small balanced data)

4. **SQL tab** (`http://localhost:4040/SQL/`)
   - Click the query — see the **visual DAG**
   - Find the **Exchange** node — this IS the shuffle, visually
   - Everything above Exchange = Stage 1; everything below = Stage 2

Run the next cell, then go back to the UI and refresh — you'll see a **new job** appear (because `.count()` is a separate action).


In [6]:
print(f"Partitions in 'sales': {sales.rdd.getNumPartitions()}")

# This is a SEPARATE action -> a SEPARATE job in the UI, even though
# it reuses the same transformations as city_totals
print(f"\nRow count: {city_totals.count()}")
print("\n✅ Refresh http://localhost:4040/jobs/ — you should now see 2 jobs total")


Partitions in 'sales': 15

Row count: 3

✅ Refresh http://localhost:4040/jobs/ — you should now see 2 jobs total


---
## 4. 🔍 Spark UI Deep Dive #2 — Caching and the Storage Tab

By default, every action re-executes the full plan from the data source. `.cache()` (or `.persist()`) tells Spark to **materialise** a DataFrame in memory after its first computation, so subsequent actions skip re-computation.

This is your first taste of caching — full depth comes in **Day 18 (Performance Tuning)**.

### What changes in the UI when you cache:
- **Storage tab** populates — shows the cached DataFrame, its size, and % cached
- Subsequent jobs show **fewer stages** — cached stages are skipped (shown as skipped/greyed in the DAG)


In [9]:
# Without cache: every action re-reads/re-computes from scratch
filtered = sales.filter(F.col("revenue") > 200)

print("Running count() WITHOUT cache (job 3):")
print(f"  Count: {filtered.count()}")

print("\nRunning count() AGAIN on same DataFrame (job 4) — recomputed from scratch:")
print(f"  Count: {filtered.count()}")

print("\n👀 Check Jobs tab: jobs 3 and 4 both have the same number of stages — no skipping yet")


Running count() WITHOUT cache (job 3):
  Count: 7

Running count() AGAIN on same DataFrame (job 4) — recomputed from scratch:
  Count: 7

👀 Check Jobs tab: jobs 3 and 4 both have the same number of stages — no skipping yet


In [10]:
# Now cache it
filtered.cache()

print("First action AFTER cache() — this computes AND stores the result (job 5):")
print(f"  Count: {filtered.count()}")

print("\n👀 Check the STORAGE tab now (http://localhost:4040/storage/)")
print("    You should see 'filtered' listed with size and 100% cached")

print("\nSecond action on the SAME cached DataFrame (job 6) — should be much faster:")
print(f"  Count: {filtered.count()}")

print("\n👀 Compare job 5 vs job 6 duration in the Jobs tab.")
print("    Job 6 should show fewer/skipped stages — cached data is reused, not recomputed.")


First action AFTER cache() — this computes AND stores the result (job 5):
  Count: 7

👀 Check the STORAGE tab now (http://localhost:4040/storage/)
    You should see 'filtered' listed with size and 100% cached

Second action on the SAME cached DataFrame (job 6) — should be much faster:
  Count: 7

👀 Compare job 5 vs job 6 duration in the Jobs tab.
    Job 6 should show fewer/skipped stages — cached data is reused, not recomputed.


In [11]:
# Always clean up cached data when done with it (frees executor memory)
filtered.unpersist()
print("Unpersisted. Check Storage tab — should now be empty.")


Unpersisted. Check Storage tab — should now be empty.


---
## 5. Structured Streaming — A Glimpse

Structured Streaming applies the **same DataFrame API** to unbounded data. The mental model: a streaming DataFrame is a table that keeps growing, and Spark incrementally computes results as new data arrives.

We won't run a real stream today (Part V, Days 19–22 are dedicated to this), but here's the key insight using a **batch** example that mirrors what a streaming job looks like:


In [12]:
# This is BATCH code — but notice it's IDENTICAL in shape to what
# Structured Streaming code looks like. Only the source/sink differ.

# Simulating "new data arriving" as a static DataFrame
events = spark.createDataFrame([
    ("2024-01-01 10:00:00", "click",  "user_1"),
    ("2024-01-01 10:00:05", "click",  "user_2"),
    ("2024-01-01 10:00:10", "purchase", "user_1"),
    ("2024-01-01 10:00:15", "click",  "user_3"),
    ("2024-01-01 10:00:20", "purchase", "user_2"),
], schema=["event_time", "event_type", "user_id"])

# This aggregation logic is EXACTLY what you'd write for a streaming job
event_counts = (
    events
    .groupBy("event_type")
    .agg(F.count("*").alias("event_count"))
    .orderBy("event_type")
)

event_counts.show()

print("In Structured Streaming, this same .groupBy().agg() logic runs")
print("incrementally as new rows arrive — same code, different read/write.")
print("Full deep dive: Days 19-22")


+----------+-----------+
|event_type|event_count|
+----------+-----------+
|     click|          3|
|  purchase|          2|
+----------+-----------+

In Structured Streaming, this same .groupBy().agg() logic runs
incrementally as new rows arrive — same code, different read/write.
Full deep dive: Days 19-22


---
## 6. MLlib — A Glimpse

MLlib provides distributed ML algorithms. Since we're doing a **deep dive** on MLlib (Days 23–28), today is just enough to recognise the shape of the API: **Transformers**, **Estimators**, and **Pipelines**.


In [14]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# Sample data: predicting revenue from units sold and discount %
training_data = spark.createDataFrame([
    (10, 0.05, 950.0),
    (20, 0.10, 1800.0),
    (15, 0.00, 1500.0),
    (30, 0.15, 2550.0),
    (25, 0.05, 2375.0),
], schema=["units_sold", "discount_pct", "revenue"])

# TRANSFORMER: VectorAssembler combines feature columns into one vector column
# (MLlib algorithms require features as a single 'vector' column)
assembler = VectorAssembler(inputCols=["units_sold", "discount_pct"], outputCol="features")
prepared = assembler.transform(training_data)
prepared.show()

# ESTIMATOR: LinearRegression — has a .fit() method that produces a Model (a Transformer)
lr = LinearRegression(featuresCol="features", labelCol="revenue")
model = lr.fit(prepared)   # .fit() = Estimator -> Model

print(f"Coefficients: {model.coefficients}")
print(f"Intercept: {model.intercept:.2f}")

# The fitted MODEL is itself a Transformer — it has .transform()
predictions = model.transform(prepared)
predictions.select("units_sold", "discount_pct", "revenue", "prediction").show()


+----------+------------+-------+-----------+
|units_sold|discount_pct|revenue|   features|
+----------+------------+-------+-----------+
|        10|        0.05|  950.0|[10.0,0.05]|
|        20|         0.1| 1800.0| [20.0,0.1]|
|        15|         0.0| 1500.0| [15.0,0.0]|
|        30|        0.15| 2550.0|[30.0,0.15]|
|        25|        0.05| 2375.0|[25.0,0.05]|
+----------+------------+-------+-----------+

Coefficients: [92.88888888888879,-2277.7777777777715]
Intercept: 136.67
+----------+------------+-------+------------------+
|units_sold|discount_pct|revenue|        prediction|
+----------+------------+-------+------------------+
|        10|        0.05|  950.0| 951.6666666666674|
|        20|         0.1| 1800.0|1766.6666666666667|
|        15|         0.0| 1500.0|            1530.0|
|        30|        0.15| 2550.0| 2581.666666666666|
|        25|        0.05| 2375.0|2344.9999999999986|
+----------+------------+-------+------------------+



**The pattern you just saw is the foundation of every MLlib pipeline:**

```
Transformer:  .transform(df) -> df    (stateless or pre-fitted)
Estimator:    .fit(df) -> Model        (learns from data)
Model:        IS a Transformer         (has .transform())
Pipeline:     chains multiple stages   (Days 23-28 cover this fully)
```

We'll build full pipelines with cross-validation, hyperparameter grids, and custom transformers starting Day 23.


---
## 7. Lower-Level APIs — RDDs (Preview)

Every DataFrame operation compiles down to RDD operations under the hood. You rarely need RDDs directly for structured data, but they're essential for:
- Custom partitioning logic
- Unstructured data (text processing, custom binary formats)
- Fine-grained control over data placement

Full coverage: **Days 11–13**. Quick preview:


In [15]:
# Get the underlying RDD from a DataFrame
rdd = sales.rdd
print(f"Type: {type(rdd)}")

# RDDs operate on Row objects directly - no Catalyst optimization
# This map() is a narrow transformation, just like DataFrame's withColumn
high_value = rdd.map(lambda row: (row.city, row.revenue)).filter(lambda x: x[1] > 500)

print(f"High value sales (city, revenue): {high_value.collect()}")

print("\n⚠️ Note: RDD operations bypass Catalyst — no query optimization.")
print("   DataFrames should be your default. RDDs are the exception, not the rule.")


Type: <class 'pyspark.core.rdd.RDD'>
High value sales (city, revenue): [('Toronto', 1200), ('Montreal', 900), ('Calgary', 700), ('Toronto', 800)]

⚠️ Note: RDD operations bypass Catalyst — no query optimization.
   DataFrames should be your default. RDDs are the exception, not the rule.


---
## 8. The Broader Ecosystem

A quick map of what exists outside core Spark (covered in more depth in **Day 29**):

| Tool | Purpose |
|------|---------|
| **Delta Lake** | ACID transactions + versioning on top of Parquet (very common in production) |
| **GraphFrames** | Graph algorithms on DataFrames (PageRank, connected components) — Day 28 |
| **Koalas / pandas API on Spark** | pandas-compatible API that runs on Spark — built into PySpark since 3.2 |
| **spark-nlp, MLflow, Great Expectations** | Common third-party additions for NLP, ML lifecycle, data quality |

We'll touch `pyspark.pandas` (the built-in pandas API) briefly in Day 29 since you're coming from a Python background — it's often the easiest on-ramp for pandas users.


---
## Summary

| Concept | Key takeaway |
|---------|-------------|
| `spark-submit` | Production execution mode — same code, no interactive REPL |
| Dataset vs DataFrame | DataFrame = `Dataset[Row]`; Python only has DataFrames |
| **Job** | One per action (count, show, collect, write) |
| **Stage** | One per shuffle boundary; visible in UI as separate task groups |
| **Cache** | Materialises a DataFrame; subsequent jobs skip recomputation (Storage tab) |
| Structured Streaming | Same DataFrame API, incremental execution — Days 19-22 |
| MLlib pattern | Transformer (`.transform()`) / Estimator (`.fit()`) / Pipeline — Days 23-28 |
| RDDs | Underlying every DataFrame; bypass Catalyst — Days 11-13 |

### Spark UI tabs covered so far
- ✅ Jobs (one per action)
- ✅ Stages (one per shuffle)
- ✅ Tasks (one per partition)
- ✅ SQL (visual DAG with Exchange = shuffle)
- ✅ Storage (cached DataFrames)
- ⏳ Environment, Executors — coming in Day 14 (How Spark Runs on a Cluster)

---

## Exercises

Move on to `day02_exercises.ipynb`.


In [16]:
spark.stop()
print("SparkSession stopped.")


SparkSession stopped.
